# ANN vs SNN 对比曲线（论文用）

在 **Kaggle** 上：把 **ANN** 与 **SNN** 的 TensorBoard 日志各自放在 **两个文件夹** 里（每夹 **只放一个** `events.out.tfevents.*`，避免合并串台）。填好路径后 **Run All**。

输出（`/kaggle/working/paper_compare/`）：
- **`fig_compare_val_acc.png`** — 验证准确率（`VAL_ACC_PLOT_MODE` 控制 raw / 平滑 / 仅 SNN 平滑等）
- **`fig_compare_train_acc.png` / `fig_compare_train_loss.png`** — `TRAIN_*_PLOT_MODE` + `TRAIN_SMOOTH`
- **`fig_compare_spike_val.png`** — SNN `spike/val`（兼容 `spike/val_global_rate`）
- **`table_sparse_ops_proxy.csv`** — ANN vs SNN（MSF / **LIF 占位**）的 **ρ×T 运算代理** 表（非焦耳）
- **`compare_metrics.csv`** / **`compare_summary.json`**

**训练建议（追到 ~95% val、控制过拟合）** 见最后一格 Markdown。

In [ ]:
!pip -q install matplotlib seaborn tensorboard pandas

In [ ]:
from __future__ import annotations

import json
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update(
    {
        "font.family": "serif",
        "font.size": 10,
        "axes.labelsize": 11,
        "legend.fontsize": 9,
        "figure.dpi": 120,
        "savefig.dpi": 300,
        "savefig.bbox": "tight",
    }
)

OUT = Path("/kaggle/working/paper_compare")
OUT.mkdir(parents=True, exist_ok=True)

In [ ]:
# ========== 每目录仅一个 tfevents 文件 ==========
# 若文件都在同一目录，可复制到两个子目录后再填路径：
#   mkdir -p /kaggle/working/tb_ann /kaggle/working/tb_snn
#   cp events_ann... /kaggle/working/tb_ann/
#   cp events_snn... /kaggle/working/tb_snn/

DIR_ANN = Path("/kaggle/working/tb_ann")
DIR_SNN = Path("/kaggle/working/tb_snn")

MAX_EPOCH = 50  # 最多画前 N 个 epoch（0 .. N-1；若某 run 更短则截断到最短）
VAL_SMOOTH = 5  # 滑动平均窗口（建议奇数）；1 = 不平滑
TRAIN_SMOOTH = 5  # 训练 acc/loss 用；1 = 不平滑

# val acc 图：raw | smooth_only（仅 MA）| both（原始淡色 + MA）| snn_smooth_only（ANN 原始 + 仅 SNN MA）
VAL_ACC_PLOT_MODE = "smooth_only"
# 训练曲线：raw | smooth_only | both
TRAIN_ACC_PLOT_MODE = "both"
TRAIN_LOSS_PLOT_MODE = "both"

LABEL_ANN = "ANN (ResNet-18)"
LABEL_SNN = "SNN (MSF)"

In [ ]:
def load_run(logdir: Path, max_step: int) -> dict:
    ea = EventAccumulator(str(logdir), size_guidance={"scalars": 0})
    ea.Reload()
    tags = set(ea.Tags().get("scalars", []))
    out = {"tags": sorted(tags)}
    for tag in (
        "loss/train",
        "acc/train",
        "acc/val",
        "spike/val",
        "spike/val_global_rate",
        "spike/train",
        "spike/train_global_rate",
    ):
        if tag not in tags:
            out[tag] = []
            continue
        out[tag] = [(int(s.step), float(s.value)) for s in ea.Scalars(tag) if int(s.step) < max_step]
    return out


def _to_series(pairs: list) -> pd.Series:
    if not pairs:
        return pd.Series(dtype=float)
    df = pd.DataFrame(pairs, columns=["epoch", "v"]).drop_duplicates("epoch", keep="last")
    return df.set_index("epoch")["v"].sort_index()


def rolling(x: pd.Series, k: int) -> pd.Series:
    if k <= 1 or len(x) < k:
        return x
    return x.rolling(k, min_periods=1, center=True).mean()


def _pick_spike_val_series(d: dict) -> pd.Series:
    for k in ("spike/val", "spike/val_global_rate"):
        s = _to_series(d.get(k, []))
        if len(s):
            return s
    return pd.Series(dtype=float)


def _ma(a, k):
    k = max(1, int(k))
    return pd.Series(a).rolling(k, min_periods=1, center=True).mean().values


def plot_compare_val(ax, x, ann_v, snn_v, mode: str, k_smooth: int, label_ann, label_snn):
    k = max(1, int(k_smooth))
    av_m, sv_m = _ma(ann_v, k), _ma(snn_v, k)
    if mode == "raw" or k <= 1:
        ax.plot(x, ann_v, "o-", ms=3, lw=1.4, color="#1b9e77", label=label_ann)
        ax.plot(x, snn_v, "s-", ms=3, lw=1.4, color="#d95f02", label=label_snn)
    elif mode == "smooth_only":
        ax.plot(x, av_m, "-", lw=2.0, color="#1b9e77", label=f"{label_ann} MA-{k}")
        ax.plot(x, sv_m, "-", lw=2.0, color="#d95f02", label=f"{label_snn} MA-{k}")
    elif mode == "snn_smooth_only":
        ax.plot(x, ann_v, "o-", ms=2.5, lw=1.2, color="#1b9e77", alpha=0.9, label=label_ann)
        ax.plot(x, sv_m, "-", lw=2.0, color="#d95f02", label=f"{label_snn} MA-{k}")
    else:
        ax.plot(x, ann_v, "o-", ms=2.5, lw=1.0, color="#1b9e77", alpha=0.35, label=f"{label_ann} raw")
        ax.plot(x, snn_v, "s-", ms=2.5, lw=1.0, color="#d95f02", alpha=0.35, label=f"{label_snn} raw")
        ax.plot(x, av_m, "-", lw=1.9, color="#1b9e77", label=f"{label_ann} MA-{k}")
        ax.plot(x, sv_m, "-", lw=1.9, color="#d95f02", label=f"{label_snn} MA-{k}")
    ax.set_ylim(0, 102)


ann = load_run(DIR_ANN, MAX_EPOCH)
snn = load_run(DIR_SNN, MAX_EPOCH)

for name, d in (("ANN", ann), ("SNN", snn)):
    print(name, "tags:", d["tags"])
    print(" ", "acc/val len", len(d.get("acc/val", [])))

In [ ]:
def _run_len(d: dict) -> int:
    n = 0
    for tag in ("acc/val", "acc/train", "loss/train"):
        s = _to_series(d.get(tag, []))
        if len(s):
            n = max(n, int(s.index.max()) + 1)
    return n


last_e = min(_run_len(ann), _run_len(snn), MAX_EPOCH)
epochs = list(range(last_e))


def align(series: pd.Series):
    return series.reindex(epochs).astype(float)


df = pd.DataFrame({"epoch": epochs})
df["ann_val_acc"] = align(_to_series(ann.get("acc/val", []))).values * 100
df["snn_val_acc"] = align(_to_series(snn.get("acc/val", []))).values * 100
df["ann_train_acc"] = align(_to_series(ann.get("acc/train", []))).values * 100
df["snn_train_acc"] = align(_to_series(snn.get("acc/train", []))).values * 100
df["ann_train_loss"] = align(_to_series(ann.get("loss/train", []))).values
df["snn_train_loss"] = align(_to_series(snn.get("loss/train", []))).values
spk = _pick_spike_val_series(snn)
if len(spk):
    df["snn_spike_val"] = align(spk).values
else:
    df["snn_spike_val"] = np.nan

df.to_csv(OUT / "compare_metrics.csv", index=False)

summ = {
    "ann": {"best_val_acc_pct": float(np.nanmax(df["ann_val_acc"])) if len(df) else None},
    "snn": {"best_val_acc_pct": float(np.nanmax(df["snn_val_acc"])) if len(df) else None},
    "epochs_plotted": len(df),
}
with open(OUT / "compare_summary.json", "w", encoding="utf-8") as f:
    json.dump(summ, f, indent=2)
print(json.dumps(summ, indent=2))

In [ ]:
x = df["epoch"].values
ann_v, snn_v = df["ann_val_acc"].values, df["snn_val_acc"].values

fig, ax = plt.subplots(figsize=(4.2, 3.2))
plot_compare_val(ax, x, ann_v, snn_v, VAL_ACC_PLOT_MODE, VAL_SMOOTH, LABEL_ANN, LABEL_SNN)
ax.set_xlabel("Epoch")
ax.set_ylabel("Validation accuracy (%)")
ax.legend(loc="lower right", frameon=True)
ax.set_title("Validation accuracy")
fig.tight_layout()
fig.savefig(OUT / "fig_compare_val_acc.png")
plt.show()

In [ ]:
def _plot_train_pair(ax, xv, ann_y, snn_y, ylabel, title, k, mode, c1, c2):
    k = max(1, int(k))
    am, sm = _ma(ann_y, k), _ma(snn_y, k)
    if mode == "raw" or k <= 1:
        ax.plot(xv, ann_y, "o-", ms=2.5, lw=1.2, color=c1, label=LABEL_ANN)
        ax.plot(xv, snn_y, "s-", ms=2.5, lw=1.2, color=c2, label=LABEL_SNN)
    elif mode == "smooth_only":
        ax.plot(xv, am, "-", lw=2.0, color=c1, label=f"{LABEL_ANN} MA-{k}")
        ax.plot(xv, sm, "-", lw=2.0, color=c2, label=f"{LABEL_SNN} MA-{k}")
    else:
        ax.plot(xv, ann_y, "o-", ms=2, lw=1.0, color=c1, alpha=0.35, label=f"{LABEL_ANN} raw")
        ax.plot(xv, snn_y, "s-", ms=2, lw=1.0, color=c2, alpha=0.35, label=f"{LABEL_SNN} raw")
        ax.plot(xv, am, "-", lw=1.9, color=c1, label=f"{LABEL_ANN} MA-{k}")
        ax.plot(xv, sm, "-", lw=1.9, color=c2, label=f"{LABEL_SNN} MA-{k}")
    ax.set_xlabel("Epoch")
    ax.set_ylabel(ylabel)
    ax.set_title(title)


fig, ax = plt.subplots(figsize=(4.2, 3.2))
_plot_train_pair(
    ax,
    x,
    df["ann_train_acc"].values,
    df["snn_train_acc"].values,
    "Training accuracy (%)",
    "Training accuracy",
    TRAIN_SMOOTH,
    TRAIN_ACC_PLOT_MODE,
    "#66a61e",
    "#7570b3",
)
ax.set_ylim(0, 102)
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig(OUT / "fig_compare_train_acc.png")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(4.2, 3.2))
_plot_train_pair(
    ax,
    x,
    df["ann_train_loss"].values,
    df["snn_train_loss"].values,
    "Training loss",
    "Training loss",
    TRAIN_SMOOTH,
    TRAIN_LOSS_PLOT_MODE,
    "#1f77b4",
    "#ff7f0e",
)
ax.legend()
fig.tight_layout()
fig.savefig(OUT / "fig_compare_train_loss.png")
plt.show()

In [ ]:
if df["snn_spike_val"].notna().any():
    fig, ax = plt.subplots(figsize=(4.2, 2.8))
    ax.plot(x, df["snn_spike_val"], "^-", color="#e7298a", ms=3, lw=1.2)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Global spike rate (val)")
    ax.set_title("SNN: validation spike activity")
    fig.tight_layout()
    fig.savefig(OUT / "fig_compare_spike_val.png")
    plt.show()
else:
    print("No SNN spike/val in log; skip spike figure.")

In [ ]:
# 一页总图（论文双栏）
fig, axes = plt.subplots(2, 2, figsize=(7.0, 5.4))
ax00, ax01, ax10, ax11 = axes.ravel()
plot_compare_val(
    ax00,
    x,
    df["ann_val_acc"].values,
    df["snn_val_acc"].values,
    VAL_ACC_PLOT_MODE,
    VAL_SMOOTH,
    LABEL_ANN,
    LABEL_SNN,
)
ax00.set_ylabel("Val acc (%)")
ax00.set_title("(a) Validation")
ax00.legend(fontsize=6)

_plot_train_pair(
    ax01,
    x,
    df["ann_train_acc"].values,
    df["snn_train_acc"].values,
    "Train acc (%)",
    "(b) Training acc.",
    TRAIN_SMOOTH,
    TRAIN_ACC_PLOT_MODE,
    "#66a61e",
    "#7570b3",
)
ax01.legend(fontsize=6)
ax01.set_ylim(0, 102)

_plot_train_pair(
    ax10,
    x,
    df["ann_train_loss"].values,
    df["snn_train_loss"].values,
    "Train loss",
    "(c) Training loss",
    TRAIN_SMOOTH,
    TRAIN_LOSS_PLOT_MODE,
    "#1f77b4",
    "#ff7f0e",
)
ax10.legend(fontsize=6)

if df["snn_spike_val"].notna().any():
    ax11.plot(x, df["snn_spike_val"], "^-", color="#e7298a", ms=2, lw=1.0)
    ax11.set_xlabel("Epoch")
    ax11.set_ylabel("Spike rate")
    ax11.set_title("(d) SNN val spikes")
else:
    ax11.axis("off")
    ax11.text(0.5, 0.5, "(d) no spike log", ha="center", va="center")

ax00.set_xlabel("Epoch")
ax01.set_xlabel("Epoch")
fig.tight_layout()
fig.savefig(OUT / "fig_compare_panel_all.png", dpi=300)
plt.show()
print("Done. Files in", OUT)

## 稀疏性 / 「总运算量」代理 vs 能耗（写论文时的口径）

- **事件稀疏（activity sparsity）**：用 TensorBoard 里的 **全局平均脉冲率** `ρ∈[0,1]`（`SpikeCounter` 与 `spike/val`）描述「每个输出时间步上平均有多少比例发生脉冲」。
- **运算量代理（非焦耳）**：在**线性近似**下，可把「与脉冲事件成正比的有效突触活动」记为 **`ρ×T`**（相对 ANN 单次前向取 **`T=1, ρ=1`**）。这用于**横向对比 MSF / LIF / ANN**，不是芯片级 **pJ/op** 能耗。
- **真实能耗（Joule）**：需要 **硬件模型**（存算一体、ADC、静态功耗等）。若论文只做到仿真，建议在 caption 写 **synaptic activity proxy** 或 **event density**，避免把 `ρ×T` 直接叫 **Energy (J)**。

下一格生成 **`table_sparse_ops_proxy.csv`**：请把 **LIF** 行的路径与 `ρ` 在跑完实验后替换；未跑通前可保留 `None`。

In [ ]:
import sys
from pathlib import Path


def _find_plantvillage_repo() -> Path:
    """Locate repo root (directory containing utils/paper_sparse_energy.py)."""
    rel = Path("utils") / "paper_sparse_energy.py"
    candidates = [
        Path("/kaggle/working/plantvillage_snn"),
        Path.cwd(),
        Path.cwd() / "plantvillage_snn",
    ]
    for c in candidates:
        c = c.resolve()
        if (c / rel).is_file():
            return c
    for parent in [Path.cwd(), *Path.cwd().parents]:
        p = parent.resolve()
        if (p / rel).is_file():
            return p
    raise FileNotFoundError(
        "Cannot import utils: clone repo so that utils/paper_sparse_energy.py exists, "
        "then chdir to repo root or set cwd to plantvillage_snn. "
        "Kaggle default: /kaggle/working/plantvillage_snn"
    )


REPO = _find_plantvillage_repo()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

if "OUT" not in globals():
    OUT = Path("/kaggle/working/paper_compare")
    OUT.mkdir(parents=True, exist_ok=True)

from utils.paper_sparse_energy import build_ops_proxy_table

# --- 手动行：填你论文最终数字；LIF 为占位 ---
ROWS = [
    {
        "method": "ANN ResNet-18",
        "neuron": "-",
        "T": 1,
        "val_acc": 99.32,
        "rho_spike": 1.0,
        "notes": "ann_out/（无脉冲，ρ=1 表示稠密激活代理）",
    },
    {
        "method": "SNN MSF",
        "neuron": "MSF",
        "T": 4,
        "val_acc": 93.54,
        "rho_spike": 0.0297,
        "notes": "Resnet18_MSF_rect_T4_lr0.05 — ρ 建议用 best-val 对应 epoch 的 spike/val 覆盖",
    },
    {
        "method": "SNN LIF",
        "neuron": "LIF",
        "T": 4,
        "val_acc": None,
        "rho_spike": None,
        "notes": "PLACEHOLDER: snn_out/Resnet18_LIF_rect_T4_lr0.05/events.out.tfevents.*",
    },
]

df_sparse = build_ops_proxy_table(ROWS)
# 相对 ANN 的 ops 代理比（ANN 行 ops_activity_proxy=1）
base = float(df_sparse.loc[df_sparse["method"].str.contains("ANN", case=False), "ops_activity_proxy"].iloc[0])
df_sparse["vs_ann_ops_proxy"] = df_sparse["ops_activity_proxy"] / max(base, 1e-9)
df_sparse.to_csv(OUT / "table_sparse_ops_proxy.csv", index=False)
print(df_sparse.to_string(index=False))

## 怎么用 15 vs 20 epoch？想 val ~95% 又少过拟合

- **不要卡在 15**：SNN 在你现在的设定下 **15 epoch 末约 90%**，要到 **95%** 通常需要 **更长**（试 **40～80 epoch**；时间允许再上）。**20 一般仍不够**，但可做中间 checkpoint。
- **时间步 `T`**：在可接受墙钟下试 **`T=6` 或 `8`**（相对 `T=4` 往往抬上限，但更慢）。
- **防过拟合（已进仓库）**：`TrainConfig.label_smoothing`，在 SNN notebook 里加 **`cfg.label_smoothing = 0.05` 或 `0.1`**（与 ANN 对比时两边可同开，公平）。
- **正则**：已有 `weight_decay=5e-4`；若 **train≫val** 再试 **`1e-3`**。
- **早停 / 选权**：仍以 **best val** 保存；可再看 **最后 5 epoch val 均值** 是否稳定。
- **报告**：论文里写 **best val** + **曲线** +（可选）**多 seed**。

现实预期：SNN **追到与 ANN 同峰值** 有时需要 **更多 epoch 或更大 T**；若差 5～8% 主要是 **预算不足** 而非仅超参微调。